In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.8 MB/s eta 0:00:00a 0:00:01


In [ ]:
# import shutil

# shutil.make_archive(
#     "/kaggle/working/adi_trained_yolov8_model",   # 👈 perfect name
#     'zip',
#     "/kaggle/working/results"
# )

In [ ]:
# import os

# train_path = "/kaggle/input/datasets/garambharadhi/major-project/train/images"
# val_path = "/kaggle/input/datasets/garambharadhi/major-project/valid/images"

# print("Train images:", len(os.listdir(train_path)))
# print("Val images:", len(os.listdir(val_path)))

In [ ]:
from ultralytics import YOLO

# model load
model = YOLO("/kaggle/input/datasets/garambharadhi/train-val-predict/adi_trained_yolov8_model/Results/train_results/weights/best.pt")

# validation run
metrics = model.val(
    data="/kaggle/input/datasets/garambharadhi/major-project/data.yaml",
    imgsz=832,
    batch=8,
    device=0,
    workers=2,
    save_json=True,
    plots=True,
    project="/kaggle/working/results",
    name="val_results"
)
# ===== METRICS PRINT =====
print("\n🔥 ===== VALIDATION METRICS ===== 🔥")

print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP75    : {metrics.box.map75:.4f}")

print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

print("\n📊 Per-class mAP:")
for i, cls_map in enumerate(metrics.box.maps):
    print(f"Class {i} mAP50-95: {cls_map:.4f}")

print("\n✅ Done!")

In [ ]:
from ultralytics import YOLO

model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

results = model.predict(
    source="/kaggle/input/datasets/garambharadhi/major-project/test/images",
    
    imgsz=832,
    conf=0.50,      # 🔥 FINAL BEST
    iou=0.40,
    max_det=8,
    
    device=0,
    save=True,
    save_txt=True,
    save_conf=True,
    show_labels=True,
    show_conf=True,
   
    agnostic_nms=False,
    stream=False,

    project="/kaggle/working/results",
    name="predict_results",
)

print("✅ Final prediction done!")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/val_predictresults",
    "zip",
    "/kaggle/working/results"
)

print("✅ Full ZIP ready")

In [ ]:
import cv2
import math
from ultralytics import YOLO

# 1. Load trained YOLO model
model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# 2. Input video path
video_path = "/kaggle/input/datasets/garambharadhi/dron-bird-videoes2/15062910_2160_3840_60fps.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Error: input video open nahi hua")

# 3. Video properties
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Size:", width, "x", height)

# 4. Output video setup
output_path = "/kaggle/working/video1.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

if not out.isOpened():
    print("Error: output writer open nahi hua")

# 5. Previous center
prev_center = None
frame_count = 0

# For smoother speed
prev_pixel_speed = 0.0
alpha = 0.7   # smoothing factor

# 6. Process frames
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    results = model(frame, verbose=False)

    best_box = None
    best_conf = 0.0
    best_cls = None

    for result in results:
        boxes = result.boxes
        for box in boxes:
            conf = float(box.conf[0])
            if conf > best_conf:
                best_conf = conf
                best_box = box
                best_cls = int(box.cls[0])

    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box.xyxy[0])

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        current_center = (cx, cy)

        class_name = model.names[best_cls]

        # Draw box + center
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(frame, current_center, 5, (0, 0, 255), -1)

        # Object label
        cv2.putText(frame, f"Object: {class_name}", (x1, y1 - 110),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

        cv2.putText(frame, f"Conf: {best_conf:.2f}", (x1, y1 - 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        if prev_center is not None:
            prev_cx, prev_cy = prev_center

            dx = cx - prev_cx
            dy = cy - prev_cy

            # Pixel distance
            distance = math.sqrt(dx**2 + dy**2)

            # Raw pixel speed
            raw_pixel_speed = distance * fps

            # Smooth speed
            pixel_speed = alpha * prev_pixel_speed + (1 - alpha) * raw_pixel_speed
            prev_pixel_speed = pixel_speed

            if abs(dx) > abs(dy):
                direction = "Right" if dx > 0 else "Left"
            else:
                direction = "Down" if dy > 0 else "Up"

            cv2.line(frame, prev_center, current_center, (0, 255, 255), 2)

            cv2.putText(frame, f"Dir: {direction}", (x1, y1 - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (220, 220, 220), 2)

            cv2.putText(frame, f"Pixel Speed: {pixel_speed:.2f}", (x1, y1 - 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

            cv2.putText(frame, f"Pixel Dist: {distance:.2f}", (x1, y1 + 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        prev_center = current_center

    out.write(frame)

cap.release()
out.release()

print("Done ✅ Saved at:", output_path)
print("Total frames processed:", frame_count)

In [ ]:
# import cv2
# import math
# from collections import deque
# from ultralytics import YOLO

# # -----------------------------
# # 1. Load trained YOLO model
# # -----------------------------
# model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# # -----------------------------
# # 2. Input video path
# # -----------------------------
# video_path = "/kaggle/input/datasets/garambharadhi/bird-dron-video/4462852-uhd_3840_2160_25fps.mp4"
# cap = cv2.VideoCapture(video_path)

# if not cap.isOpened():
#     print("Error: input video open nahi hua")

# # -----------------------------
# # 3. Video properties
# # -----------------------------
# fps = cap.get(cv2.CAP_PROP_FPS)
# if fps == 0:
#     fps = 30

# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# print("FPS:", fps)
# print("Size:", width, "x", height)

# # -----------------------------
# # 4. Output video setup
# # -----------------------------
# output_path = "/kaggle/working/physics_informed_output_fixed_panels_bigtext.mp4"
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# if not out.isOpened():
#     print("Error: output writer open nahi hua")

# # -----------------------------
# # 5. Tunable thresholds
# # -----------------------------
# STABILITY_THRESHOLD = 28.0
# LOW_CONF_THRESHOLD = 0.80
# ANGLE_BIRD_THRESHOLD = 12.0
# ACC_BIRD_THRESHOLD = 25.0
# BIRD_SCORE_MARGIN = 2.0

# # -----------------------------
# # 6. Variables
# # -----------------------------
# prev_center = None
# prev_pixel_speed = 0.0
# prev_angle = None
# frame_count = 0

# alpha = 0.7

# speed_history = deque(maxlen=10)
# acc_history = deque(maxlen=10)
# angle_change_history = deque(maxlen=10)
# area_history = deque(maxlen=10)

# def get_direction(dx, dy):
#     if abs(dx) > abs(dy):
#         return "Right" if dx > 0 else "Left"
#     else:
#         return "Down" if dy > 0 else "Up"

# def draw_panel(frame, x=20, y=20, w=760, h=620):
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)
#     cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
#     cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 3)

# def draw_panel_text(frame, text, x, y, color=(255, 255, 255), scale=1.25, thickness=3):
#     cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)

# # -----------------------------
# # 7. Process frames
# # -----------------------------
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame_count += 1
#     results = model(frame, verbose=False)

#     best_box = None
#     best_conf = 0.0
#     best_cls = None

#     for result in results:
#         boxes = result.boxes
#         for box in boxes:
#             conf = float(box.conf[0])
#             if conf > best_conf:
#                 best_conf = conf
#                 best_box = box
#                 best_cls = int(box.cls[0])

#     yolo_label = "None"
#     final_label = "None"
#     motion_label = "No Detection"
#     direction = "N/A"
#     distance = 0.0
#     pixel_speed = 0.0
#     acceleration = 0.0
#     angle_change = 0.0
#     bbox_area = 0
#     stability_score = 0.0
#     speed_var = 0.0
#     acc_var = 0.0
#     avg_angle_change = 0.0
#     area_var = 0.0

#     if best_box is not None:
#         x1, y1, x2, y2 = map(int, best_box.xyxy[0])

#         cx = (x1 + x2) // 2
#         cy = (y1 + y2) // 2
#         current_center = (cx, cy)

#         yolo_label = model.names[best_cls]
#         final_label = yolo_label
#         bbox_area = (x2 - x1) * (y2 - y1)

#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 4)
#         cv2.circle(frame, current_center, 8, (0, 0, 255), -1)

#         if prev_center is not None:
#             prev_cx, prev_cy = prev_center

#             dx = cx - prev_cx
#             dy = cy - prev_cy

#             distance = math.sqrt(dx**2 + dy**2)

#             raw_pixel_speed = distance * fps
#             pixel_speed = alpha * prev_pixel_speed + (1 - alpha) * raw_pixel_speed

#             acceleration = pixel_speed - prev_pixel_speed
#             prev_pixel_speed = pixel_speed

#             direction = get_direction(dx, dy)

#             current_angle = math.degrees(math.atan2(dy, dx))
#             if prev_angle is not None:
#                 angle_change = abs(current_angle - prev_angle)
#                 if angle_change > 180:
#                     angle_change = 360 - angle_change
#             prev_angle = current_angle

#             cv2.line(frame, prev_center, current_center, (0, 255, 255), 4)

#             speed_history.append(pixel_speed)
#             acc_history.append(abs(acceleration))
#             angle_change_history.append(angle_change)
#             area_history.append(bbox_area)

#             speed_var = max(speed_history) - min(speed_history) if len(speed_history) > 1 else 0.0
#             acc_var = sum(acc_history) / len(acc_history) if len(acc_history) > 0 else 0.0
#             avg_angle_change = sum(angle_change_history) / len(angle_change_history) if len(angle_change_history) > 0 else 0.0
#             area_var = max(area_history) - min(area_history) if len(area_history) > 1 else 0.0

#             stability_score = (
#                 0.30 * speed_var +
#                 0.30 * acc_var +
#                 0.25 * avg_angle_change +
#                 0.15 * (area_var / 1000.0)
#             )

#             if stability_score >= STABILITY_THRESHOLD:
#                 motion_label = "Irregular Motion"
#                 motion_based_class = "Bird"
#             else:
#                 motion_label = "Stable Motion"
#                 motion_based_class = "Drone"

#             if yolo_label.lower() == motion_based_class.lower():
#                 final_label = yolo_label
#             else:
#                 if (
#                     motion_based_class == "Bird" and
#                     (
#                         stability_score >= (STABILITY_THRESHOLD - BIRD_SCORE_MARGIN) or
#                         avg_angle_change >= ANGLE_BIRD_THRESHOLD or
#                         acc_var >= ACC_BIRD_THRESHOLD or
#                         best_conf < LOW_CONF_THRESHOLD
#                     )
#                 ):
#                     final_label = "Bird"
#                 else:
#                     final_label = yolo_label

#         prev_center = current_center

#     # -----------------------------
#     # 8. Fixed top-left big info panel
#     # -----------------------------
#     draw_panel(frame, x=20, y=20, w=760, h=620)

#     if final_label.lower() == "drone":
#         final_color = (0, 255, 0)
#     elif final_label.lower() == "bird":
#         final_color = (0, 0, 255)
#     else:
#         final_color = (255, 255, 255)

#     draw_panel_text(frame, "PHYSICS-INFORMED ANALYSIS", 45, 75, color=(255, 255, 255), scale=1.35, thickness=3)
#     draw_panel_text(frame, f"YOLO Label: {yolo_label}", 45, 135, color=(0, 255, 255), scale=1.20, thickness=3)
#     draw_panel_text(frame, f"Final Label: {final_label}", 45, 195, color=final_color, scale=1.35, thickness=4)
#     draw_panel_text(frame, f"Confidence: {best_conf:.2f}", 45, 255, color=(255, 255, 0), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"Direction: {direction}", 45, 315, color=(220, 220, 220), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"Pixel Speed: {pixel_speed:.2f}", 45, 375, color=(0, 165, 255), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"Acceleration: {acceleration:.2f}", 45, 435, color=(255, 150, 0), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"Angle Change: {angle_change:.2f}", 45, 495, color=(255, 100, 255), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"BBox Area: {bbox_area}", 45, 555, color=(100, 255, 255), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"Motion State: {motion_label}", 45, 615, color=(255, 100, 100), scale=1.10, thickness=3)

#     out.write(frame)

# cap.release()
# out.release()

# print("Done ✅ Saved at:", output_path)
# print("Total frames processed:", frame_count)
# print("Used Stability Threshold:", STABILITY_THRESHOLD)

In [ ]:
# import cv2
# import math
# from collections import deque
# from ultralytics import YOLO

# # -----------------------------
# # 1. Load trained YOLO model
# # -----------------------------
# model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# # -----------------------------
# # 2. Input video path
# # -----------------------------
# video_path = "/kaggle/input/datasets/garambharadhi/dron-bird-videoes2/12886447_2160_3840_30fps.mp4"
# cap = cv2.VideoCapture(video_path)

# if not cap.isOpened():
#     print("Error: input video open nahi hua")

# # -----------------------------
# # 3. Video properties
# # -----------------------------
# fps = cap.get(cv2.CAP_PROP_FPS)
# if fps == 0:
#     fps = 30

# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# print("FPS:", fps)
# print("Size:", width, "x", height)

# # -----------------------------
# # 4. Output video setup
# # -----------------------------
# output_path = "/kaggle/working/videoo.mp4"
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# if not out.isOpened():
#     print("Error: output writer open nahi hua")

# # -----------------------------
# # 5. Variables
# # -----------------------------
# prev_center = None
# prev_pixel_speed = 0.0
# prev_angle = None
# frame_count = 0
# alpha = 0.7

# speed_history = deque(maxlen=10)
# acc_history = deque(maxlen=10)
# angle_change_history = deque(maxlen=10)
# area_history = deque(maxlen=10)

# # Only YOLO temporal smoothing for final class
# label_history = deque(maxlen=10)
# stable_final_label = "None"

# def get_direction(dx, dy):
#     if abs(dx) > abs(dy):
#         return "Right" if dx > 0 else "Left"
#     else:
#         return "Down" if dy > 0 else "Up"

# def draw_panel(frame, x=20, y=20, w=760, h=620):
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)
#     cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
#     cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 3)

# def draw_panel_text(frame, text, x, y, color=(255, 255, 255), scale=1.15, thickness=3):
#     cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)

# # -----------------------------
# # 6. Process frames
# # -----------------------------
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame_count += 1
#     results = model(frame, verbose=False)

#     best_box = None
#     best_conf = 0.0
#     best_cls = None

#     for result in results:
#         boxes = result.boxes
#         for box in boxes:
#             conf = float(box.conf[0])
#             if conf > best_conf:
#                 best_conf = conf
#                 best_box = box
#                 best_cls = int(box.cls[0])

#     # defaults
#     yolo_label = "None"
#     final_label = "None"
#     motion_label = "No Detection"
#     direction = "N/A"
#     pixel_speed = 0.0
#     acceleration = 0.0
#     angle_change = 0.0
#     bbox_area = 0
#     stability_score = 0.0
#     avg_angle_change = 0.0
#     acc_var = 0.0

#     if best_box is not None:
#         x1, y1, x2, y2 = map(int, best_box.xyxy[0])

#         cx = (x1 + x2) // 2
#         cy = (y1 + y2) // 2
#         current_center = (cx, cy)

#         yolo_label = model.names[best_cls]
#         final_label = yolo_label
#         bbox_area = (x2 - x1) * (y2 - y1)

#         # Draw object
#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 4)
#         cv2.circle(frame, current_center, 8, (0, 0, 255), -1)

#         if prev_center is not None:
#             prev_cx, prev_cy = prev_center

#             dx = cx - prev_cx
#             dy = cy - prev_cy

#             distance = math.sqrt(dx**2 + dy**2)

#             raw_pixel_speed = distance * fps
#             pixel_speed = alpha * prev_pixel_speed + (1 - alpha) * raw_pixel_speed

#             acceleration = pixel_speed - prev_pixel_speed
#             prev_pixel_speed = pixel_speed

#             direction = get_direction(dx, dy)

#             current_angle = math.degrees(math.atan2(dy, dx))
#             if prev_angle is not None:
#                 angle_change = abs(current_angle - prev_angle)
#                 if angle_change > 180:
#                     angle_change = 360 - angle_change
#             prev_angle = current_angle

#             cv2.line(frame, prev_center, current_center, (0, 255, 255), 4)

#             speed_history.append(pixel_speed)
#             acc_history.append(abs(acceleration))
#             angle_change_history.append(angle_change)
#             area_history.append(bbox_area)

#             speed_var = max(speed_history) - min(speed_history) if len(speed_history) > 1 else 0.0
#             acc_var = sum(acc_history) / len(acc_history) if len(acc_history) > 0 else 0.0
#             avg_angle_change = sum(angle_change_history) / len(angle_change_history) if len(angle_change_history) > 0 else 0.0
#             area_var = max(area_history) - min(area_history) if len(area_history) > 1 else 0.0

#             stability_score = (
#                 0.30 * speed_var +
#                 0.30 * acc_var +
#                 0.25 * avg_angle_change +
#                 0.15 * (area_var / 1000.0)
#             )

#             if stability_score >= 32:
#                 motion_label = "Irregular Motion"
#             else:
#                 motion_label = "Stable Motion"

#         prev_center = current_center

#         # -----------------------------
#         # Only YOLO smoothing for final class
#         # -----------------------------
#         if yolo_label in ["Bird", "Drone"]:
#             label_history.append(yolo_label)

#         bird_votes = sum(1 for lbl in label_history if lbl == "Bird")
#         drone_votes = sum(1 for lbl in label_history if lbl == "Drone")

#         if bird_votes >= 6:
#             stable_final_label = "Bird"
#         elif drone_votes >= 6:
#             stable_final_label = "Drone"
#         else:
#             stable_final_label = yolo_label

#         final_label = stable_final_label

#     else:
#         # reset when no detection
#         prev_center = None
#         prev_angle = None
#         prev_pixel_speed = 0.0
#         speed_history.clear()
#         acc_history.clear()
#         angle_change_history.clear()
#         area_history.clear()
#         label_history.clear()
#         stable_final_label = "None"

#     # -----------------------------
#     # 7. Info panel
#     # -----------------------------
#     draw_panel(frame, x=20, y=20, w=760, h=620)

#     if final_label.lower() == "drone":
#         final_color = (0, 255, 0)
#     elif final_label.lower() == "bird":
#         final_color = (0, 0, 255)
#     else:
#         final_color = (255, 255, 255)

#     draw_panel_text(frame, "PHYSICS-INFORMED ANALYSIS", 45, 75, color=(255, 255, 255), scale=1.30, thickness=3)
#     draw_panel_text(frame, f"YOLO Label: {yolo_label}", 45, 135, color=(0, 255, 255), scale=1.15, thickness=3)
#     draw_panel_text(frame, f"Final Label: {final_label}", 45, 195, color=final_color, scale=1.30, thickness=4)
#     draw_panel_text(frame, f"Confidence: {best_conf:.2f}", 45, 255, color=(255, 255, 0), scale=1.05, thickness=3)
#     draw_panel_text(frame, f"Direction: {direction}", 45, 315, color=(220, 220, 220), scale=1.05, thickness=3)
#     draw_panel_text(frame, f"Pixel Speed: {pixel_speed:.2f}", 45, 375, color=(0, 165, 255), scale=1.05, thickness=3)
#     draw_panel_text(frame, f"Acceleration: {acceleration:.2f}", 45, 435, color=(255, 150, 0), scale=1.05, thickness=3)
#     draw_panel_text(frame, f"Angle Change: {angle_change:.2f}", 45, 495, color=(255, 100, 255), scale=1.05, thickness=3)
#     draw_panel_text(frame, f"BBox Area: {bbox_area}", 45, 555, color=(100, 255, 255), scale=1.05, thickness=3)
#     draw_panel_text(frame, f"Motion State: {motion_label}", 45, 615, color=(255, 100, 100), scale=1.05, thickness=3)

#     out.write(frame)

# cap.release()
# out.release()

# print("Done ✅ Saved at:", output_path)
# print("Total frames processed:", frame_count)

In [ ]:
# import cv2
# import math
# from collections import deque
# from ultralytics import YOLO

# # -----------------------------
# # 1. Load trained YOLO model
# # -----------------------------
# model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# # -----------------------------
# # 2. Input video path
# # -----------------------------
# video_path = "/kaggle/input/datasets/garambharadhi/dron-bird-videoes2/15062910_2160_3840_60fps.mp4"
# cap = cv2.VideoCapture(video_path)

# if not cap.isOpened():
#     print("Error: input video open nahi hua")

# # -----------------------------
# # 3. Video properties
# # -----------------------------
# fps = cap.get(cv2.CAP_PROP_FPS)
# if fps == 0:
#     fps = 30

# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# print("FPS:", fps)
# print("Size:", width, "x", height)

# # -----------------------------
# # 4. Output video setup
# # -----------------------------
# output_path = "/kaggle/working/final_integrated_output.mp4"
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# if not out.isOpened():
#     print("Error: output writer open nahi hua")

# # -----------------------------
# # 5. Parameters
# # -----------------------------
# alpha = 0.7

# # trajectory settings
# trajectory = deque(maxlen=20)
# motion_history = deque(maxlen=10)

# # label smoothing
# label_history = deque(maxlen=10)
# stable_final_label = "None"

# # override rules
# LOW_CONF_THRESHOLD = 0.80
# STRONG_BIRD_RATIO = 0.50
# STRONG_DRONE_RATIO = 0.90

# # tracking variables
# prev_center = None
# prev_pixel_speed = 0.0
# frame_count = 0

# # -----------------------------
# # 6. Helper functions
# # -----------------------------
# def get_direction(dx, dy):
#     if abs(dx) > abs(dy):
#         return "Right" if dx > 0 else "Left"
#     else:
#         return "Down" if dy > 0 else "Up"

# def draw_panel(frame, x=20, y=20, w=860, h=560):
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)
#     cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
#     cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 3)

# def draw_panel_text(frame, text, x, y, color=(255, 255, 255), scale=1.00, thickness=3):
#     cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)

# # -----------------------------
# # 7. Process frames
# # -----------------------------
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame_count += 1
#     results = model(frame, verbose=False)

#     best_box = None
#     best_conf = 0.0
#     best_cls = None

#     for result in results:
#         boxes = result.boxes
#         for box in boxes:
#             conf = float(box.conf[0])
#             if conf > best_conf:
#                 best_conf = conf
#                 best_box = box
#                 best_cls = int(box.cls[0])

#     # default values
#     yolo_label = "None"
#     final_label = "None"
#     direction = "N/A"
#     pixel_speed = 0.0
#     distance = 0.0
#     smoothness_ratio = 0.0
#     motion_label = "No Detection"
#     override_status = "No Override"
#     combined_insight = "No object detected."

#     if best_box is not None:
#         x1, y1, x2, y2 = map(int, best_box.xyxy[0])

#         cx = (x1 + x2) // 2
#         cy = (y1 + y2) // 2
#         current_center = (cx, cy)

#         yolo_label = model.names[best_cls]
#         final_label = yolo_label  # base = YOLO

#         # Draw object
#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
#         cv2.circle(frame, current_center, 6, (0, 0, 255), -1)

#         # Store trajectory
#         trajectory.append(current_center)

#         # Draw trajectory
#         for i in range(1, len(trajectory)):
#             cv2.line(frame, trajectory[i - 1], trajectory[i], (0, 255, 255), 2)
#             cv2.circle(frame, trajectory[i], 3, (255, 0, 255), -1)

#         # pixel speed + direction
#         if prev_center is not None:
#             prev_cx, prev_cy = prev_center
#             dx = cx - prev_cx
#             dy = cy - prev_cy

#             distance = math.sqrt(dx**2 + dy**2)
#             raw_pixel_speed = distance * fps
#             pixel_speed = alpha * prev_pixel_speed + (1 - alpha) * raw_pixel_speed
#             prev_pixel_speed = pixel_speed

#             direction = get_direction(dx, dy)

#         # -----------------------------
#         # Trajectory smoothness
#         # -----------------------------
#         if len(trajectory) >= 6:
#             total_distance = 0.0

#             for i in range(1, len(trajectory)):
#                 x_prev, y_prev = trajectory[i - 1]
#                 x_curr, y_curr = trajectory[i]
#                 total_distance += math.hypot(x_curr - x_prev, y_curr - y_prev)

#             x_start, y_start = trajectory[0]
#             x_end, y_end = trajectory[-1]
#             straight_distance = math.hypot(x_end - x_start, y_end - y_start)

#             smoothness_ratio = straight_distance / (total_distance + 1e-6)

#             if smoothness_ratio >= 0.82:
#                 motion_label = "Drone-like Motion"
#             elif smoothness_ratio <= 0.60:
#                 motion_label = "Bird-like Motion"
#             else:
#                 motion_label = "Mixed Motion"

#             motion_history.append(motion_label)

#             # stable motion voting
#             drone_votes = sum(1 for m in motion_history if m == "Drone-like Motion")
#             bird_votes = sum(1 for m in motion_history if m == "Bird-like Motion")
#             mixed_votes = sum(1 for m in motion_history if m == "Mixed Motion")

#             if drone_votes >= max(bird_votes, mixed_votes):
#                 stable_motion_label = "Drone-like Motion"
#             elif bird_votes >= max(drone_votes, mixed_votes):
#                 stable_motion_label = "Bird-like Motion"
#             else:
#                 stable_motion_label = "Mixed Motion"

#             # -----------------------------
#             # Selective override logic
#             # -----------------------------
#             if best_conf < LOW_CONF_THRESHOLD:
#                 if smoothness_ratio < STRONG_BIRD_RATIO:
#                     final_label = "Bird"
#                     override_status = "Selective Override -> Bird"
#                 elif smoothness_ratio > STRONG_DRONE_RATIO:
#                     final_label = "Drone"
#                     override_status = "Selective Override -> Drone"
#                 else:
#                     override_status = "YOLO kept (trajectory not strong enough)"
#             else:
#                 override_status = "YOLO kept (high confidence)"

#             # insight text
#             if yolo_label.lower() == final_label.lower():
#                 if final_label == "Bird":
#                     combined_insight = "Final decision stays BIRD."
#                 elif final_label == "Drone":
#                     combined_insight = "Final decision stays DRONE."
#                 else:
#                     combined_insight = "Final decision unchanged."
#             else:
#                 combined_insight = f"YOLO corrected using trajectory pattern -> {final_label}"

#         else:
#             stable_motion_label = "Analyzing..."
#             combined_insight = "Collecting enough trajectory points..."

#         prev_center = current_center

#         # -----------------------------
#         # Final label smoothing
#         # -----------------------------
#         if final_label in ["Bird", "Drone"]:
#             label_history.append(final_label)

#         bird_label_votes = sum(1 for lbl in label_history if lbl == "Bird")
#         drone_label_votes = sum(1 for lbl in label_history if lbl == "Drone")

#         if bird_label_votes >= 6:
#             stable_final_label = "Bird"
#         elif drone_label_votes >= 6:
#             stable_final_label = "Drone"
#         else:
#             stable_final_label = final_label

#         final_label = stable_final_label

#     else:
#         prev_center = None
#         prev_pixel_speed = 0.0
#         trajectory.clear()
#         motion_history.clear()
#         label_history.clear()
#         stable_final_label = "None"

#     # -----------------------------
#     # 8. Fixed premium panel
#     # -----------------------------
#     draw_panel(frame, x=20, y=20, w=860, h=560)

#     if final_label.lower() == "drone":
#         final_color = (0, 255, 0)
#     elif final_label.lower() == "bird":
#         final_color = (0, 0, 255)
#     else:
#         final_color = (255, 255, 255)

#     draw_panel_text(frame, "PHYSICS-INFORMED DRONE/BIRD ANALYSIS", 45, 70, color=(255, 255, 255), scale=1.10, thickness=3)
#     draw_panel_text(frame, f"YOLO Label: {yolo_label}", 45, 125, color=(0, 255, 255), scale=0.95, thickness=3)
#     draw_panel_text(frame, f"Final Label: {final_label}", 45, 180, color=final_color, scale=1.10, thickness=4)
#     draw_panel_text(frame, f"Confidence: {best_conf:.2f}", 45, 235, color=(255, 255, 0), scale=0.90, thickness=3)
#     draw_panel_text(frame, f"Direction: {direction}", 45, 290, color=(220, 220, 220), scale=0.90, thickness=3)
#     draw_panel_text(frame, f"Pixel Speed: {pixel_speed:.2f}", 45, 345, color=(0, 165, 255), scale=0.90, thickness=3)
#     draw_panel_text(frame, f"Pixel Distance: {distance:.2f}", 45, 400, color=(255, 100, 100), scale=0.90, thickness=3)
#     draw_panel_text(frame, f"Smoothness Ratio: {smoothness_ratio:.2f}", 45, 455, color=(100, 255, 100), scale=0.90, thickness=3)
#     draw_panel_text(frame, f"Trajectory Motion: {motion_label}", 45, 510, color=(255, 100, 255), scale=0.90, thickness=3)

#     draw_panel_text(frame, f"Override Status: {override_status}", 430, 125, color=(255, 255, 255), scale=0.85, thickness=3)
#     draw_panel_text(frame, "Combined Insight:", 430, 190, color=(255, 255, 255), scale=0.90, thickness=3)
#     draw_panel_text(frame, combined_insight[:45], 430, 245, color=(255, 255, 255), scale=0.80, thickness=2)
#     if len(combined_insight) > 45:
#         draw_panel_text(frame, combined_insight[45:90], 430, 285, color=(255, 255, 255), scale=0.80, thickness=2)

#     out.write(frame)

# cap.release()
# out.release()

# print("Done ✅ Saved at:", output_path)
# print("Total frames processed:", frame_count)
# print("Low confidence threshold:", LOW_CONF_THRESHOLD)
# print("Strong bird ratio:", STRONG_BIRD_RATIO)
# print("Strong drone ratio:", STRONG_DRONE_RATIO)

In [ ]:
# import cv2
# import math
# import numpy as np
# from collections import deque, Counter
# from ultralytics import YOLO

# # -----------------------------
# # 1. Load trained YOLO model
# # -----------------------------
# model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# # -----------------------------
# # 2. Input / Output paths
# # -----------------------------
# video_path = "/kaggle/input/datasets/garambharadhi/dron-bird-videoes2/19005328-uhd_1440_2560_60fps.mp4"
# output_path = "/kaggle/working/adaptive_physics_informed_output.mp4"

# cap = cv2.VideoCapture(video_path)
# if not cap.isOpened():
#     raise ValueError("Input video open nahi hua")

# fps = cap.get(cv2.CAP_PROP_FPS)
# if fps == 0:
#     fps = 30

# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
# if not out.isOpened():
#     raise ValueError("Output writer open nahi hua")

# print("FPS:", fps)
# print("Size:", width, "x", height)

# # -----------------------------
# # 3. Tracking + histories
# # -----------------------------
# track_history = deque(maxlen=20)   # center history
# speed_history = deque(maxlen=15)
# acc_history = deque(maxlen=15)
# angle_history = deque(maxlen=15)
# area_history = deque(maxlen=15)
# label_history = deque(maxlen=15)
# conf_history = deque(maxlen=15)

# prev_center = None
# prev_speed = None
# prev_angle = None

# # -----------------------------
# # 4. Helper functions
# # -----------------------------
# def get_center(box):
#     x1, y1, x2, y2 = box
#     return ((x1 + x2) // 2, (y1 + y2) // 2)

# def euclidean(p1, p2):
#     return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# def clamp(x, low=0.0, high=1.0):
#     return max(low, min(high, x))

# def safe_std(values):
#     return float(np.std(values)) if len(values) > 1 else 0.0

# def safe_mean(values):
#     return float(np.mean(values)) if len(values) > 0 else 0.0

# def compute_angle(p1, p2):
#     dx = p2[0] - p1[0]
#     dy = p2[1] - p1[1]
#     return math.degrees(math.atan2(dy, dx))

# def angle_diff(a, b):
#     d = abs(a - b)
#     return min(d, 360 - d)

# def majority_vote(labels):
#     if len(labels) == 0:
#         return "Unknown"
#     return Counter(labels).most_common(1)[0][0]

# def draw_panel(frame, lines, x=20, y=20, w=700, h=420):
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)
#     cv2.addWeighted(overlay, 0.58, frame, 0.42, 0, frame)
#     cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 2)

#     yy = y + 35
#     for text, color, scale in lines:
#         cv2.putText(frame, text, (x + 15, yy), cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2)
#         yy += 35

# # -----------------------------
# # 5. Adaptive scoring function
# # -----------------------------
# def classify_motion(speed_hist, acc_hist, angle_hist, area_hist, yolo_conf, yolo_label):
#     """
#     No hard threshold style final decision.
#     Uses normalized feature scores + temporal logic.
#     """

#     speed_std = safe_std(speed_hist)
#     acc_mean = safe_mean(np.abs(acc_hist)) if len(acc_hist) > 0 else 0.0
#     angle_mean = safe_mean(angle_hist)
#     area_std = safe_std(area_hist)

#     # Normalize features to 0-1 range
#     # Ye values empirical hain, but hard cutoff nahi ban rahe
#     speed_score = clamp(speed_std / 35.0)
#     acc_score   = clamp(acc_mean / 20.0)
#     angle_score = clamp(angle_mean / 25.0)
#     area_score  = clamp(area_std / 2500.0)

#     # Bird irregularity score
#     bird_motion_score = (
#         0.30 * speed_score +
#         0.25 * acc_score +
#         0.30 * angle_score +
#         0.15 * area_score
#     )

#     # Drone smoothness score = inverse of irregularity
#     drone_motion_score = 1.0 - bird_motion_score

#     # YOLO confidence contribution
#     yolo_bird = 0.0
#     yolo_drone = 0.0

#     if yolo_label.lower() == "bird":
#         yolo_bird = yolo_conf
#         yolo_drone = 1.0 - yolo_conf
#     elif yolo_label.lower() == "drone":
#         yolo_drone = yolo_conf
#         yolo_bird = 1.0 - yolo_conf
#     else:
#         yolo_bird = 0.5
#         yolo_drone = 0.5

#     # Final fusion
#     final_bird_score = 0.55 * bird_motion_score + 0.45 * yolo_bird
#     final_drone_score = 0.55 * drone_motion_score + 0.45 * yolo_drone

#     final_label = "Bird" if final_bird_score > final_drone_score else "Drone"

#     return {
#         "final_label": final_label,
#         "bird_motion_score": bird_motion_score,
#         "drone_motion_score": drone_motion_score,
#         "final_bird_score": final_bird_score,
#         "final_drone_score": final_drone_score,
#         "speed_std": speed_std,
#         "acc_mean": acc_mean,
#         "angle_mean": angle_mean,
#         "area_std": area_std
#     }

# # -----------------------------
# # 6. Main loop
# # -----------------------------
# frame_count = 0

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame_count += 1

#     # YOLO tracking
#     results = model.track(
#         source=frame,
#         persist=True,
#         tracker="bytetrack.yaml",
#         conf=0.35,
#         iou=0.45,
#         verbose=False
#     )

#     best_box = None
#     best_conf = 0.0
#     best_cls = None

#     if results and len(results) > 0:
#         result = results[0]
#         if result.boxes is not None and len(result.boxes) > 0:
#             for box in result.boxes:
#                 conf = float(box.conf[0])
#                 cls_id = int(box.cls[0])

#                 if conf > best_conf:
#                     best_conf = conf
#                     best_cls = cls_id
#                     xyxy = box.xyxy[0].cpu().numpy().astype(int)
#                     best_box = xyxy

#     yolo_label = "None"
#     final_label = "No Detection"

#     speed = 0.0
#     acceleration = 0.0
#     angle_change = 0.0
#     area = 0

#     if best_box is not None:
#         x1, y1, x2, y2 = best_box
#         cx, cy = get_center(best_box)
#         center = (cx, cy)
#         area = max(1, (x2 - x1) * (y2 - y1))

#         yolo_label = model.names[best_cls]

#         # Draw detection
#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#         cv2.circle(frame, center, 4, (0, 0, 255), -1)

#         track_history.append(center)
#         area_history.append(area)
#         conf_history.append(best_conf)

#         if prev_center is not None:
#             dist = euclidean(prev_center, center)
#             speed = dist * fps
#             speed_history.append(speed)

#             if prev_speed is not None:
#                 acceleration = speed - prev_speed
#                 acc_history.append(acceleration)
#             prev_speed = speed

#             current_angle = compute_angle(prev_center, center)
#             if prev_angle is not None:
#                 angle_change = angle_diff(current_angle, prev_angle)
#                 angle_history.append(angle_change)
#             prev_angle = current_angle

#             # draw path
#             for i in range(1, len(track_history)):
#                 cv2.line(frame, track_history[i - 1], track_history[i], (0, 255, 255), 2)

#         prev_center = center

#         # classify only when enough motion history is available
#         if len(speed_history) >= 5 and len(angle_history) >= 4 and len(area_history) >= 5:
#             result_scores = classify_motion(
#                 speed_history,
#                 acc_history,
#                 angle_history,
#                 area_history,
#                 best_conf,
#                 yolo_label
#             )

#             current_label = result_scores["final_label"]
#             label_history.append(current_label)
#             final_label = majority_vote(label_history)

#             bird_motion_score = result_scores["bird_motion_score"]
#             drone_motion_score = result_scores["drone_motion_score"]
#             final_bird_score = result_scores["final_bird_score"]
#             final_drone_score = result_scores["final_drone_score"]
#             speed_std = result_scores["speed_std"]
#             acc_mean = result_scores["acc_mean"]
#             angle_mean = result_scores["angle_mean"]
#             area_std = result_scores["area_std"]
#         else:
#             final_label = yolo_label
#             bird_motion_score = 0.0
#             drone_motion_score = 0.0
#             final_bird_score = 0.0
#             final_drone_score = 0.0
#             speed_std = 0.0
#             acc_mean = 0.0
#             angle_mean = 0.0
#             area_std = 0.0

#         color = (0, 255, 0) if final_label.lower() == "drone" else (0, 0, 255)
#         cv2.putText(frame, f"Final: {final_label}", (x1, max(30, y1 - 10)),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

#         lines = [
#             ("PHYSICS-INFORMED VIDEO CLASSIFICATION", (255, 255, 255), 0.8),
#             (f"YOLO Label: {yolo_label}", (0, 255, 255), 0.7),
#             (f"YOLO Conf: {best_conf:.2f}", (255, 255, 0), 0.7),
#             (f"Final Label: {final_label}", color, 0.8),
#             (f"Speed: {speed:.2f}", (0, 165, 255), 0.65),
#             (f"Acceleration: {acceleration:.2f}", (255, 150, 0), 0.65),
#             (f"Angle Change: {angle_change:.2f}", (255, 100, 255), 0.65),
#             (f"Area: {area}", (100, 255, 255), 0.65),
#             (f"Speed Std: {speed_std:.2f}", (220, 220, 220), 0.6),
#             (f"Acc Mean: {acc_mean:.2f}", (220, 220, 220), 0.6),
#             (f"Angle Mean: {angle_mean:.2f}", (220, 220, 220), 0.6),
#             (f"Area Std: {area_std:.2f}", (220, 220, 220), 0.6),
#             (f"Bird Score: {final_bird_score:.2f}", (0, 0, 255), 0.65),
#             (f"Drone Score: {final_drone_score:.2f}", (0, 255, 0), 0.65),
#         ]
#         draw_panel(frame, lines, x=20, y=20, w=520, h=500)

#     else:
#         lines = [
#             ("PHYSICS-INFORMED VIDEO CLASSIFICATION", (255, 255, 255), 0.8),
#             ("No object detected", (0, 0, 255), 0.75)
#         ]
#         draw_panel(frame, lines, x=20, y=20, w=520, h=100)

#     out.write(frame)

# cap.release()
# out.release()

# print("Done ✅ Saved at:", output_path)
# print("Total frames processed:", frame_count)

In [ ]:
# import cv2
# import math
# import numpy as np
# from collections import deque, Counter
# from ultralytics import YOLO

# # =========================================================
# # 1. LOAD MODEL
# # =========================================================
# model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# # =========================================================
# # 2. INPUT / OUTPUT PATHS
# # =========================================================
# video_path = "/kaggle/input/datasets/garambharadhi/bird-dron-video/4462852-uhd_3840_2160_25fps.mp4"
# output_path = "/kaggle/working/live_adaptive_drone_bird_output.mp4"

# cap = cv2.VideoCapture(video_path)
# if not cap.isOpened():
#     raise ValueError("Error: input video open nahi hua")

# fps = cap.get(cv2.CAP_PROP_FPS)
# if fps == 0 or np.isnan(fps):
#     fps = 30.0

# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
# if not out.isOpened():
#     raise ValueError("Error: output writer open nahi hua")

# print("FPS:", fps)
# print("Frame Size:", width, "x", height)

# # =========================================================
# # 3. PARAMETERS
# # =========================================================
# TRACK_HISTORY_LEN = 30
# WINDOW_SIZE = 20
# LABEL_VOTE_WINDOW = 15
# WARMUP_FRAMES = 20

# YOLO_CONF_USE = 0.35
# YOLO_IOU_USE = 0.45

# # =========================================================
# # 4. HISTORIES
# # =========================================================
# track_points = deque(maxlen=TRACK_HISTORY_LEN)

# speed_hist = deque(maxlen=WINDOW_SIZE)
# acc_hist = deque(maxlen=WINDOW_SIZE)
# angle_hist = deque(maxlen=WINDOW_SIZE)
# area_hist = deque(maxlen=WINDOW_SIZE)
# conf_hist = deque(maxlen=WINDOW_SIZE)

# label_vote_hist = deque(maxlen=LABEL_VOTE_WINDOW)
# video_labels = []

# prev_center = None
# prev_speed = None
# prev_angle = None

# frame_count = 0

# # =========================================================
# # 5. HELPER FUNCTIONS
# # =========================================================
# def clamp(x, low=0.0, high=1.0):
#     return max(low, min(high, x))

# def safe_mean(arr):
#     return float(np.mean(arr)) if len(arr) > 0 else 0.0

# def safe_std(arr):
#     return float(np.std(arr)) if len(arr) > 1 else 0.0

# def euclidean(p1, p2):
#     return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# def compute_angle(p1, p2):
#     dx = p2[0] - p1[0]
#     dy = p2[1] - p1[1]
#     return math.degrees(math.atan2(dy, dx))

# def angle_diff(a, b):
#     diff = abs(a - b)
#     return min(diff, 360 - diff)

# def get_direction(dx, dy):
#     if abs(dx) > abs(dy):
#         return "Right" if dx > 0 else "Left"
#     else:
#         return "Down" if dy > 0 else "Up"

# def majority_vote(labels):
#     if len(labels) == 0:
#         return "Unknown"
#     return Counter(labels).most_common(1)[0][0]

# def draw_panel(frame, lines, x=20, y=20, w=760, h=520):
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)
#     cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
#     cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 2)

#     yy = y + 40
#     for text, color, scale, thickness in lines:
#         cv2.putText(
#             frame, text, (x + 15, yy),
#             cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness
#         )
#         yy += int(32 + scale * 8)

# def dynamic_thresholds(speed_hist, acc_hist, angle_hist, area_hist):
#     speed_mean = safe_mean(speed_hist)
#     speed_std = safe_std(speed_hist)

#     acc_abs = [abs(x) for x in acc_hist]
#     acc_mean = safe_mean(acc_abs)
#     acc_std = safe_std(acc_abs)

#     angle_mean = safe_mean(angle_hist)
#     angle_std = safe_std(angle_hist)

#     area_mean = safe_mean(area_hist)
#     area_std = safe_std(area_hist)

#     # Dynamic thresholds from live video statistics
#     speed_thr = speed_mean + 1.0 * speed_std
#     acc_thr = acc_mean + 1.0 * acc_std
#     angle_thr = angle_mean + 1.0 * angle_std
#     area_thr = area_mean + 0.8 * area_std

#     return {
#         "speed_mean": speed_mean,
#         "speed_std": speed_std,
#         "acc_mean": acc_mean,
#         "acc_std": acc_std,
#         "angle_mean": angle_mean,
#         "angle_std": angle_std,
#         "area_mean": area_mean,
#         "area_std": area_std,
#         "speed_thr": speed_thr,
#         "acc_thr": acc_thr,
#         "angle_thr": angle_thr,
#         "area_thr": area_thr
#     }

# def motion_score(
#     current_speed,
#     current_acc,
#     current_angle_change,
#     current_area,
#     dyn_stats
# ):
#     """
#     Bird irregularity score vs Drone smoothness score
#     Live adaptive thresholds used from current video history
#     """

#     # Avoid division by zero
#     speed_thr = max(dyn_stats["speed_thr"], 1e-6)
#     acc_thr = max(dyn_stats["acc_thr"], 1e-6)
#     angle_thr = max(dyn_stats["angle_thr"], 1e-6)
#     area_thr = max(dyn_stats["area_thr"], 1e-6)

#     # Normalize against live thresholds
#     speed_score = clamp(current_speed / speed_thr)
#     acc_score = clamp(abs(current_acc) / acc_thr)
#     angle_score = clamp(current_angle_change / angle_thr)
#     area_score = clamp(current_area / area_thr)

#     # Bird = irregular
#     bird_irregularity = (
#         0.25 * speed_score +
#         0.25 * acc_score +
#         0.30 * angle_score +
#         0.20 * area_score
#     )

#     # Drone = stable/smooth
#     drone_smoothness = 1.0 - bird_irregularity

#     return bird_irregularity, drone_smoothness

# def fuse_with_yolo(yolo_label, yolo_conf, bird_motion_score, drone_motion_score):
#     yolo_bird = 0.5
#     yolo_drone = 0.5

#     if str(yolo_label).lower() == "bird":
#         yolo_bird = yolo_conf
#         yolo_drone = 1.0 - yolo_conf
#     elif str(yolo_label).lower() == "drone":
#         yolo_drone = yolo_conf
#         yolo_bird = 1.0 - yolo_conf

#     final_bird_score = 0.55 * bird_motion_score + 0.45 * yolo_bird
#     final_drone_score = 0.55 * drone_motion_score + 0.45 * yolo_drone

#     final_label = "Bird" if final_bird_score > final_drone_score else "Drone"
#     return final_label, final_bird_score, final_drone_score

# # =========================================================
# # 6. MAIN LOOP
# # =========================================================
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame_count += 1

#     # -----------------------------------------------------
#     # YOLO TRACK
#     # -----------------------------------------------------
#     results = model.track(
#         source=frame,
#         persist=True,
#         tracker="bytetrack.yaml",
#         conf=YOLO_CONF_USE,
#         iou=YOLO_IOU_USE,
#         verbose=False
#     )

#     best_box = None
#     best_conf = 0.0
#     best_cls = None

#     if results and len(results) > 0:
#         result = results[0]
#         if result.boxes is not None and len(result.boxes) > 0:
#             for box in result.boxes:
#                 conf = float(box.conf[0])
#                 cls_id = int(box.cls[0])

#                 if conf > best_conf:
#                     xyxy = box.xyxy[0].cpu().numpy().astype(int)
#                     x1, y1, x2, y2 = xyxy

#                     # invalid boxes skip
#                     if x2 <= x1 or y2 <= y1:
#                         continue

#                     best_box = (x1, y1, x2, y2)
#                     best_conf = conf
#                     best_cls = cls_id

#     # -----------------------------------------------------
#     # DEFAULT DISPLAY VALUES
#     # -----------------------------------------------------
#     yolo_label = "None"
#     final_label = "No Detection"
#     direction = "N/A"

#     current_speed = 0.0
#     current_acc = 0.0
#     current_angle_change = 0.0
#     current_area_var = 0.0
#     bbox_area = 0

#     bird_motion_score = 0.0
#     drone_motion_score = 0.0
#     final_bird_score = 0.0
#     final_drone_score = 0.0

#     dyn = {
#         "speed_mean": 0.0, "speed_std": 0.0,
#         "acc_mean": 0.0, "acc_std": 0.0,
#         "angle_mean": 0.0, "angle_std": 0.0,
#         "area_mean": 0.0, "area_std": 0.0,
#         "speed_thr": 1.0, "acc_thr": 1.0,
#         "angle_thr": 1.0, "area_thr": 1.0
#     }

#     # -----------------------------------------------------
#     # PROCESS BEST BOX
#     # -----------------------------------------------------
#     if best_box is not None:
#         x1, y1, x2, y2 = best_box
#         cx = (x1 + x2) // 2
#         cy = (y1 + y2) // 2
#         current_center = (cx, cy)

#         yolo_label = model.names[best_cls]
#         bbox_area = (x2 - x1) * (y2 - y1)

#         # draw box
#         cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
#         cv2.circle(frame, current_center, 4, (0, 0, 255), -1)

#         track_points.append(current_center)
#         area_hist.append(bbox_area)
#         conf_hist.append(best_conf)

#         if prev_center is not None:
#             dx = current_center[0] - prev_center[0]
#             dy = current_center[1] - prev_center[1]
#             direction = get_direction(dx, dy)

#             dist = euclidean(prev_center, current_center)
#             current_speed = dist * fps
#             speed_hist.append(current_speed)

#             if prev_speed is not None:
#                 current_acc = current_speed - prev_speed
#                 acc_hist.append(current_acc)

#             current_angle = compute_angle(prev_center, current_center)
#             if prev_angle is not None:
#                 current_angle_change = angle_diff(current_angle, prev_angle)
#                 angle_hist.append(current_angle_change)

#             prev_angle = current_angle
#             prev_speed = current_speed

#         else:
#             prev_speed = 0.0

#         prev_center = current_center

#         # draw trajectory
#         if len(track_points) >= 2:
#             for i in range(1, len(track_points)):
#                 cv2.line(frame, track_points[i - 1], track_points[i], (0, 255, 255), 2)

#         # current area variation from live window
#         if len(area_hist) > 1:
#             current_area_var = max(area_hist) - min(area_hist)
#         else:
#             current_area_var = 0.0

#         # -------------------------------------------------
#         # CLASSIFICATION AFTER WARMUP
#         # -------------------------------------------------
#         enough_history = (
#             frame_count > WARMUP_FRAMES and
#             len(speed_hist) >= 5 and
#             len(acc_hist) >= 3 and
#             len(angle_hist) >= 3 and
#             len(area_hist) >= 5
#         )

#         if enough_history:
#             dyn = dynamic_thresholds(speed_hist, acc_hist, angle_hist, area_hist)

#             bird_motion_score, drone_motion_score = motion_score(
#                 current_speed=current_speed,
#                 current_acc=current_acc,
#                 current_angle_change=current_angle_change,
#                 current_area=current_area_var,
#                 dyn_stats=dyn
#             )

#             current_frame_label, final_bird_score, final_drone_score = fuse_with_yolo(
#                 yolo_label=yolo_label,
#                 yolo_conf=best_conf,
#                 bird_motion_score=bird_motion_score,
#                 drone_motion_score=drone_motion_score
#             )

#             label_vote_hist.append(current_frame_label)
#             final_label = majority_vote(label_vote_hist)
#         else:
#             # Warmup phase: YOLO label only
#             if str(yolo_label).lower() in ["bird", "drone"]:
#                 final_label = str(yolo_label).capitalize()
#             else:
#                 final_label = "Unknown"

#         video_labels.append(final_label)

#         # label color
#         if final_label.lower() == "drone":
#             label_color = (0, 255, 0)
#         elif final_label.lower() == "bird":
#             label_color = (0, 0, 255)
#         else:
#             label_color = (255, 255, 255)

#         # box label
#         cv2.putText(
#             frame,
#             f"{final_label} ({best_conf:.2f})",
#             (x1, max(30, y1 - 10)),
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.9,
#             label_color,
#             2
#         )

#         # panel lines
#         lines = [
#             ("LIVE ADAPTIVE PHYSICS-INFORMED CLASSIFICATION", (255, 255, 255), 0.75, 2),
#             (f"Frame: {frame_count}", (220, 220, 220), 0.68, 2),
#             (f"YOLO Label: {yolo_label}", (0, 255, 255), 0.70, 2),
#             (f"Final Label: {final_label}", label_color, 0.85, 2),
#             (f"Confidence: {best_conf:.2f}", (255, 255, 0), 0.68, 2),
#             (f"Direction: {direction}", (200, 200, 200), 0.65, 2),
#             (f"Speed: {current_speed:.2f}", (0, 165, 255), 0.62, 2),
#             (f"Acceleration: {current_acc:.2f}", (255, 140, 0), 0.62, 2),
#             (f"Angle Change: {current_angle_change:.2f}", (255, 100, 255), 0.62, 2),
#             (f"Area Variation: {current_area_var:.2f}", (100, 255, 255), 0.62, 2),
#             (f"Live Speed Thr: {dyn['speed_thr']:.2f}", (180, 180, 180), 0.58, 1),
#             (f"Live Acc Thr: {dyn['acc_thr']:.2f}", (180, 180, 180), 0.58, 1),
#             (f"Live Angle Thr: {dyn['angle_thr']:.2f}", (180, 180, 180), 0.58, 1),
#             (f"Live Area Thr: {dyn['area_thr']:.2f}", (180, 180, 180), 0.58, 1),
#             (f"Bird Motion Score: {bird_motion_score:.2f}", (0, 0, 255), 0.62, 2),
#             (f"Drone Motion Score: {drone_motion_score:.2f}", (0, 255, 0), 0.62, 2),
#             (f"Bird Final Score: {final_bird_score:.2f}", (0, 0, 255), 0.62, 2),
#             (f"Drone Final Score: {final_drone_score:.2f}", (0, 255, 0), 0.62, 2),
#         ]
#         draw_panel(frame, lines, x=20, y=20, w=780, h=560)

#     else:
#         prev_center = None
#         prev_speed = None
#         prev_angle = None
#         track_points.clear()

#         lines = [
#             ("LIVE ADAPTIVE PHYSICS-INFORMED CLASSIFICATION", (255, 255, 255), 0.75, 2),
#             ("No detection in current frame", (0, 0, 255), 0.80, 2),
#         ]
#         draw_panel(frame, lines, x=20, y=20, w=620, h=100)

#     out.write(frame)

# # =========================================================
# # 7. FINAL VIDEO-LEVEL RESULT
# # =========================================================
# cap.release()
# out.release()

# if len(video_labels) > 0:
#     video_counter = Counter(video_labels)
#     final_video_label = video_counter.most_common(1)[0][0]
# else:
#     final_video_label = "Unknown"

# print("\nDone ✅")
# print("Saved at:", output_path)
# print("Total frames processed:", frame_count)
# print("Final video classification:", final_video_label)
# print("Video label distribution:", dict(Counter(video_labels)))

In [22]:
import requests

ESP32_URL = "https://malka-unfrosty-grady.ngrok-free.dev"

def esp_on():
    try:
        requests.get(f"{ESP32_URL}/drone", timeout=5)
        print("ESP ON")
    except Exception as e:
        print("ESP error:", e)

def esp_off():
    try:
        requests.get(f"{ESP32_URL}/clear", timeout=5)
        print("ESP OFF")
    except Exception as e:
        print("ESP clear error:", e)

In [23]:
import requests
import time

BOT_TOKEN = "8627260755:AAEUX6FAa4SGaDrrm9FGt0FQYLdUsZBMcwM"
CHAT_ID = "5969400352"

ESP32_URL = "https://malka-unfrosty-grady.ngrok-free.dev"

last_alert_time = 0
COOLDOWN = 10   # seconds

def send_telegram(msg):
    try:
        url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
        data = {"chat_id": CHAT_ID, "text": msg}
        requests.post(url, data=data)
    except:
        pass

def esp_on():
    try:
        requests.get(f"{ESP32_URL}/drone", timeout=5)
    except:
        pass

def esp_off():
    try:
        requests.get(f"{ESP32_URL}/clear", timeout=5)
    except:
        pass

def send_drone_alert():
    global last_alert_time
    now = time.time()

    if now - last_alert_time < COOLDOWN:
        return

    print("🚨 DRONE DETECTED")
    send_telegram("🚨 Drone Detected!")
    esp_on()   # sirf drone pe ESP ON
    last_alert_time = now

def send_bird_alert():
    global last_alert_time
    now = time.time()

    if now - last_alert_time < COOLDOWN:
        return

    print("🕊️ BIRD DETECTED")
    send_telegram("🕊️ Bird Detected!")
    last_alert_time = now

In [24]:
import requests

BOT_TOKEN = "8627260755:AAEUX6FAa4SGaDrrm9FGt0FQYLdUsZBMcwM"
CHAT_ID = "5969400352"

url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
data = {
    "chat_id": CHAT_ID,
    "text": "Test message from drone project"
}

r = requests.post(url, data=data, timeout=10)
print(r.status_code)
print(r.text)

200
{"ok":true,"result":{"message_id":4,"from":{"id":8627260755,"is_bot":true,"first_name":"DroneBirdAdiBot","username":"DroneBirdAdiBot"},"chat":{"id":5969400352,"first_name":"Aditya","last_name":"Karan","type":"private"},"date":1778011316,"text":"Test message from drone project"}}


In [29]:
import cv2
import math
import numpy as np
from collections import deque, Counter
from ultralytics import YOLO

# =========================================================
# 1. LOAD MODEL
# =========================================================
model = YOLO("/kaggle/input/datasets/garambharadhi/trained-model/train_results/weights/best.pt")

# =========================================================
# 2. INPUT / OUTPUT
# =========================================================
video_path = "/kaggle/input/datasets/garambharadhi/bird-dron-video3/8482187-hd_1920_1080_30fps.mp4"
output_path = "/kaggle/working/final_crop_recheck_output.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError("Input video open nahi hua")

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or np.isnan(fps):
    fps = 30.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
if not out.isOpened():
    raise ValueError("Output writer open nahi hua")

print("FPS:", fps)
print("Frame Size:", width, "x", height)

# =========================================================
# 3. PARAMETERS
# =========================================================
YOLO_CONF_USE = 0.25
YOLO_IOU_USE = 0.45

TRACK_HISTORY_LEN = 30
WINDOW_SIZE = 20
FINAL_VOTE_WINDOW = 15
YOLO_VOTE_WINDOW = 12
RECHECK_VOTE_WINDOW = 10
MISSING_FRAME_HOLD = 8
WARMUP_FRAMES = 10

MAX_TRACK_DIST = 250
MAX_CAMERA_MOTION = 15.0

MAX_CORNERS = 200
QUALITY_LEVEL = 0.01
MIN_DISTANCE = 20
BLOCK_SIZE = 3

MIN_VALID_SPEED = 5.0
MAX_VALID_SPEED = 1000.0

VERY_HIGH_CONF = 0.85
HIGH_CONF = 0.70
MID_CONF = 0.55

DRONE_SCORE_MIN = 0.72
BIRD_SCORE_MIN = 0.58
UNCERTAIN_GAP = 0.10

# crop recheck
CROP_PAD = 12
MIN_CROP_SIZE = 20

# =========================================================
# 4. STATE
# =========================================================
track_points = deque(maxlen=TRACK_HISTORY_LEN)
speed_hist = deque(maxlen=WINDOW_SIZE)
acc_hist = deque(maxlen=WINDOW_SIZE)
angle_hist = deque(maxlen=WINDOW_SIZE)
area_hist = deque(maxlen=WINDOW_SIZE)

final_vote_hist = deque(maxlen=FINAL_VOTE_WINDOW)
yolo_vote_hist = deque(maxlen=YOLO_VOTE_WINDOW)
recheck_vote_hist = deque(maxlen=RECHECK_VOTE_WINDOW)

video_labels = []

prev_center = None
prev_speed = None
prev_angle = None
prev_gray = None

frame_count = 0
missing_count = 0
last_final_label = "Unknown"
target_track_id = None

alert_active = False
drone_consecutive_frames = 0
REQUIRED_DRONE_FRAMES = 3
CLEAR_AFTER_MISSING = 8

# =========================================================
# 5. HELPERS
# =========================================================
def clamp(x, low=0.0, high=1.0):
    return max(low, min(high, x))

def safe_mean(arr):
    return float(np.mean(arr)) if len(arr) > 0 else 0.0

def safe_std(arr):
    return float(np.std(arr)) if len(arr) > 1 else 0.0

def point_dist(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def euclidean(dx, dy):
    return math.sqrt(dx * dx + dy * dy)

def compute_angle(dx, dy):
    return math.degrees(math.atan2(dy, dx))

def angle_diff(a, b):
    diff = abs(a - b)
    return min(diff, 360 - diff)

def get_direction(dx, dy):
    if abs(dx) > abs(dy):
        return "Right" if dx > 0 else "Left"
    return "Down" if dy > 0 else "Up"

def majority_vote(labels):
    if not labels:
        return "Unknown"
    return Counter(labels).most_common(1)[0][0]

def draw_panel(frame, lines, x=20, y=20, w=900, h=670):
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.60, frame, 0.40, 0, frame)
    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 255, 255), 2)

    yy = y + 35
    for text, color, scale, thickness in lines:
        cv2.putText(frame, text, (x + 15, yy),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, thickness)
        yy += int(28 + scale * 10)

def estimate_camera_motion(prev_gray, curr_gray):
    if prev_gray is None or curr_gray is None:
        return 0.0, 0.0, False

    prev_pts = cv2.goodFeaturesToTrack(
        prev_gray,
        maxCorners=MAX_CORNERS,
        qualityLevel=QUALITY_LEVEL,
        minDistance=MIN_DISTANCE,
        blockSize=BLOCK_SIZE
    )
    if prev_pts is None or len(prev_pts) < 10:
        return 0.0, 0.0, False

    curr_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, curr_gray, prev_pts, None)
    if curr_pts is None or status is None:
        return 0.0, 0.0, False

    status = status.reshape(-1)
    good_prev, good_curr = [], []

    for i in range(len(status)):
        if status[i] == 1:
            gp = np.array(prev_pts[i]).reshape(-1)
            gc = np.array(curr_pts[i]).reshape(-1)
            if gp.shape[0] >= 2 and gc.shape[0] >= 2:
                good_prev.append([gp[0], gp[1]])
                good_curr.append([gc[0], gc[1]])

    if len(good_prev) < 10:
        return 0.0, 0.0, False

    good_prev = np.array(good_prev, dtype=np.float32)
    good_curr = np.array(good_curr, dtype=np.float32)
    motion = good_curr - good_prev

    dx = float(np.median(motion[:, 0]))
    dy = float(np.median(motion[:, 1]))
    mag = math.sqrt(dx * dx + dy * dy)

    if mag > MAX_CAMERA_MOTION:
        return 0.0, 0.0, False

    return dx, dy, True

def dynamic_thresholds(speed_hist, acc_hist, angle_hist, area_hist):
    speed_mean = safe_mean(speed_hist)
    speed_std = safe_std(speed_hist)

    acc_abs = [abs(x) for x in acc_hist]
    acc_mean = safe_mean(acc_abs)
    acc_std = safe_std(acc_abs)

    angle_mean = safe_mean(angle_hist)
    angle_std = safe_std(angle_hist)

    area_mean = safe_mean(area_hist)
    area_std = safe_std(area_hist)

    return {
        "speed_thr": max(speed_mean + 1.0 * speed_std, 1e-6),
        "acc_thr": max(acc_mean + 1.0 * acc_std, 1e-6),
        "angle_thr": max(angle_mean + 1.0 * angle_std, 1e-6),
        "area_thr": max(area_mean + 0.8 * area_std, 1e-6),
    }

def motion_scores(current_speed, current_acc, current_angle_change, current_area_var, dyn):
    speed_score = clamp(current_speed / dyn["speed_thr"])
    acc_score = clamp(abs(current_acc) / dyn["acc_thr"])
    angle_score = clamp(current_angle_change / dyn["angle_thr"])
    area_score = clamp(current_area_var / dyn["area_thr"])

    bird_motion = (
        0.25 * speed_score +
        0.25 * acc_score +
        0.30 * angle_score +
        0.20 * area_score
    )
    drone_motion = 1.0 - bird_motion
    return clamp(bird_motion), clamp(drone_motion)

def extract_candidates(result_boxes, names):
    candidates = []
    if result_boxes is None or len(result_boxes) == 0:
        return candidates

    has_ids = getattr(result_boxes, "id", None) is not None
    ids = result_boxes.id.cpu().numpy().astype(int).tolist() if has_ids else [None] * len(result_boxes)

    for idx, box in enumerate(result_boxes):
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        xyxy = box.xyxy[0].cpu().numpy().astype(int)
        x1, y1, x2, y2 = xyxy
        if x2 <= x1 or y2 <= y1:
            continue

        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2

        candidates.append({
            "box": (x1, y1, x2, y2),
            "conf": conf,
            "cls_id": cls_id,
            "label": names[cls_id],
            "center": (cx, cy),
            "track_id": ids[idx] if idx < len(ids) else None
        })
    return candidates

def select_target_candidate(candidates, target_track_id, prev_center):
    if not candidates:
        return None, target_track_id

    if target_track_id is not None:
        same_id = [c for c in candidates if c["track_id"] == target_track_id]
        if same_id:
            chosen = max(same_id, key=lambda c: c["conf"])
            return chosen, target_track_id

    if prev_center is not None:
        nearby = []
        for c in candidates:
            d = point_dist(prev_center, c["center"])
            if d <= MAX_TRACK_DIST:
                score = c["conf"] - 0.0015 * d
                nearby.append((score, c))
        if nearby:
            chosen = max(nearby, key=lambda x: x[0])[1]
            return chosen, chosen["track_id"]

    chosen = max(candidates, key=lambda c: c["conf"])
    return chosen, chosen["track_id"]

def crop_recheck(frame, box):
    x1, y1, x2, y2 = box
    h, w = frame.shape[:2]

    x1p = max(0, x1 - CROP_PAD)
    y1p = max(0, y1 - CROP_PAD)
    x2p = min(w, x2 + CROP_PAD)
    y2p = min(h, y2 + CROP_PAD)

    crop = frame[y1p:y2p, x1p:x2p]
    if crop.size == 0:
        return "Unknown", 0.0

    ch, cw = crop.shape[:2]
    if ch < MIN_CROP_SIZE or cw < MIN_CROP_SIZE:
        return "Unknown", 0.0

    re_results = model.predict(crop, conf=0.05, verbose=False)
    if not re_results or re_results[0].boxes is None or len(re_results[0].boxes) == 0:
        return "Unknown", 0.0

    best = max(re_results[0].boxes, key=lambda b: float(b.conf[0]))
    conf = float(best.conf[0])
    cls_id = int(best.cls[0])
    label = model.names[cls_id]
    return label.capitalize(), conf

def final_decision(yolo_label, yolo_conf, crop_label, crop_conf, bird_motion, drone_motion, motion_reliable):
    # detector scores
    yolo_bird = 0.5
    yolo_drone = 0.5
    if yolo_label.lower() == "bird":
        yolo_bird = yolo_conf
        yolo_drone = 1.0 - yolo_conf
    elif yolo_label.lower() == "drone":
        yolo_drone = yolo_conf
        yolo_bird = 1.0 - yolo_conf

    # crop recheck scores
    crop_bird = 0.5
    crop_drone = 0.5
    if crop_label.lower() == "bird":
        crop_bird = crop_conf
        crop_drone = 1.0 - crop_conf
    elif crop_label.lower() == "drone":
        crop_drone = crop_conf
        crop_bird = 1.0 - crop_conf

    if not motion_reliable:
        det_w, crop_w, motion_w = 0.45, 0.45, 0.10
        mode = "Detector+Crop"
    else:
        det_w, crop_w, motion_w = 0.40, 0.35, 0.25
        mode = "Detector+Crop+Motion"

    bird_score = clamp(det_w * yolo_bird + crop_w * crop_bird + motion_w * bird_motion)
    drone_score = clamp(det_w * yolo_drone + crop_w * crop_drone + motion_w * drone_motion)
    gap = abs(bird_score - drone_score)

    # strict drone
    if (
        yolo_label.lower() == "drone" and crop_label.lower() == "drone" and
        yolo_conf >= MID_CONF and crop_conf >= MID_CONF and
        drone_score >= DRONE_SCORE_MIN and gap >= UNCERTAIN_GAP
    ):
        return "Drone", bird_score, drone_score, mode + "+StrictDrone", gap

    # bird
    if (
        (yolo_label.lower() == "bird" or crop_label.lower() == "bird") and
        bird_score >= BIRD_SCORE_MIN and gap >= UNCERTAIN_GAP
    ):
        return "Bird", bird_score, drone_score, mode + "+Bird", gap

    return "Uncertain", bird_score, drone_score, mode + "+Uncertain", gap

# =========================================================
# 6. MAIN LOOP
# =========================================================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    cam_dx, cam_dy, camera_motion_used = estimate_camera_motion(prev_gray, curr_gray)

    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=YOLO_CONF_USE,
        iou=YOLO_IOU_USE,
        verbose=False
    )

    chosen = None
    if results and len(results) > 0:
        candidates = extract_candidates(results[0].boxes, model.names)
        chosen, target_track_id = select_target_candidate(candidates, target_track_id, prev_center)

    yolo_label = "None"
    crop_label = "Unknown"
    final_label = "No Detection"
    direction = "N/A"
    current_speed = 0.0
    current_acc = 0.0
    current_angle_change = 0.0
    current_area_var = 0.0
    bird_motion = 0.5
    drone_motion = 0.5
    bird_score = 0.5
    drone_score = 0.5
    score_gap = 0.0
    fusion_mode = "N/A"
    motion_reliable = False
    yolo_conf = 0.0
    crop_conf = 0.0
    dyn = {"speed_thr":1.0,"acc_thr":1.0,"angle_thr":1.0,"area_thr":1.0}

    if chosen is not None:
        missing_count = 0

        box = chosen["box"]
        x1, y1, x2, y2 = box
        current_center = chosen["center"]
        yolo_conf = chosen["conf"]
        yolo_label = model.names[chosen["cls_id"]]
        bbox_area = max(1, x2 - x1) * max(1, y2 - y1)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(frame, current_center, 4, (0, 0, 255), -1)

        track_points.append(current_center)
        area_hist.append(bbox_area)

        if yolo_label.lower() in ["bird", "drone"]:
            yolo_vote_hist.append(yolo_label.capitalize())

        crop_label, crop_conf = crop_recheck(frame, box)
        if crop_label in ["Bird", "Drone"]:
            recheck_vote_hist.append(crop_label)

        if prev_center is not None:
            raw_dx = current_center[0] - prev_center[0]
            raw_dy = current_center[1] - prev_center[1]

            obj_dx = raw_dx - cam_dx
            obj_dy = raw_dy - cam_dy

            direction = get_direction(obj_dx, obj_dy)

            dist = euclidean(obj_dx, obj_dy)
            current_speed = dist * fps
            speed_hist.append(current_speed)

            if prev_speed is not None:
                current_acc = current_speed - prev_speed
                acc_hist.append(current_acc)

            if abs(obj_dx) > 1e-3 or abs(obj_dy) > 1e-3:
                current_angle = compute_angle(obj_dx, obj_dy)
                if prev_angle is not None:
                    current_angle_change = angle_diff(current_angle, prev_angle)
                    angle_hist.append(current_angle_change)
                prev_angle = current_angle

            prev_speed = current_speed
        else:
            prev_speed = 0.0

        prev_center = current_center

        if len(track_points) >= 2:
            for i in range(1, len(track_points)):
                cv2.line(frame, track_points[i - 1], track_points[i], (0, 255, 255), 2)

        if len(area_hist) > 1:
            current_area_var = max(area_hist) - min(area_hist)

        enough_history = (
            frame_count > WARMUP_FRAMES and
            len(speed_hist) >= 5 and
            len(acc_hist) >= 3 and
            len(angle_hist) >= 3 and
            len(area_hist) >= 5
        )

        if enough_history:
            dyn = dynamic_thresholds(speed_hist, acc_hist, angle_hist, area_hist)
            bird_motion, drone_motion = motion_scores(
                current_speed=current_speed,
                current_acc=current_acc,
                current_angle_change=current_angle_change,
                current_area_var=current_area_var,
                dyn=dyn
            )

            motion_reliable = True
            if current_speed < MIN_VALID_SPEED or current_speed > MAX_VALID_SPEED:
                bird_motion = 0.5
                drone_motion = 0.5
                motion_reliable = False

            if not camera_motion_used:
                bird_motion = 0.5 * bird_motion + 0.25
                drone_motion = 0.5 * drone_motion + 0.25
                motion_reliable = False

            frame_label, bird_score, drone_score, fusion_mode, score_gap = final_decision(
                yolo_label=yolo_label,
                yolo_conf=yolo_conf,
                crop_label=crop_label,
                crop_conf=crop_conf,
                bird_motion=bird_motion,
                drone_motion=drone_motion,
                motion_reliable=motion_reliable
            )

            if frame_label in ["Bird", "Drone"]:
                final_vote_hist.append(frame_label)

            voted_label = majority_vote(final_vote_hist)
            if frame_label == "Uncertain":
                final_label = voted_label if voted_label in ["Bird", "Drone"] else "Uncertain"
            else:
                final_label = frame_label
        else:
            final_label = yolo_label.capitalize() if yolo_label.lower() in ["bird", "drone"] else "Unknown"
            fusion_mode = "Warmup"

        last_final_label = final_label
        video_labels.append(final_label)
                      # ================= TELEGRAM + ESP32 ALERT LOGIC =================
        if final_label == "Drone" and drone_score >= DRONE_SCORE_MIN:
            drone_consecutive_frames += 1
        else:
            drone_consecutive_frames = 0

        # Drone: Telegram + ESP32
        if drone_consecutive_frames >= REQUIRED_DRONE_FRAMES and not alert_active:
            alert_active = True
            send_drone_alert()

        # Bird: sirf Telegram
        if final_label == "Bird" and not alert_active:
            send_bird_alert()

        # ESP32 sirf drone ke baad off hoga
        if final_label != "Drone" and alert_active and missing_count >= CLEAR_AFTER_MISSING:
            alert_active = False
            esp_off()
        # ================================================================

        if final_label.lower() == "drone":
            label_color = (0, 255, 0)
        elif final_label.lower() == "bird":
            label_color = (0, 0, 255)
        elif final_label.lower() == "uncertain":
            label_color = (0, 255, 255)
        else:
            label_color = (255, 255, 255)

        cv2.putText(frame, f"{final_label} ({yolo_conf:.2f})",
                    (x1, max(30, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, label_color, 2)

        lines = [
            ("CROP-RECHECK DRONE/BIRD CLASSIFICATION", (255,255,255), 0.68, 2),
            (f"Frame: {frame_count}", (220,220,220), 0.62, 2),
            (f"Track ID: {target_track_id}", (220,220,220), 0.58, 1),
            (f"YOLO Label: {yolo_label}", (0,255,255), 0.62, 2),
            (f"Crop Label: {crop_label}", (255,200,0), 0.62, 2),
            (f"Final Label: {final_label}", label_color, 0.78, 2),
            (f"YOLO Conf: {yolo_conf:.2f}", (255,255,0), 0.58, 2),
            (f"Crop Conf: {crop_conf:.2f}", (255,255,0), 0.58, 2),
            (f"Fusion Mode: {fusion_mode}", (255,200,100), 0.56, 2),
            (f"YOLO Majority: {majority_vote(yolo_vote_hist)}", (180,255,180), 0.56, 1),
            (f"Crop Majority: {majority_vote(recheck_vote_hist)}", (180,255,180), 0.56, 1),
            (f"Camera Motion Used: {camera_motion_used}", (180,255,180), 0.56, 1),
            (f"Direction: {direction}", (220,220,220), 0.56, 2),
            (f"Speed: {current_speed:.2f}", (0,165,255), 0.56, 2),
            (f"Acceleration: {current_acc:.2f}", (255,140,0), 0.56, 2),
            (f"Angle Change: {current_angle_change:.2f}", (255,100,255), 0.56, 2),
            (f"Area Variation: {current_area_var:.2f}", (100,255,255), 0.56, 2),
            (f"Motion Reliable: {motion_reliable}", (255,255,255), 0.54, 1),
            (f"Bird Motion Score: {bird_motion:.2f}", (0,0,255), 0.56, 2),
            (f"Drone Motion Score: {drone_motion:.2f}", (0,255,0), 0.56, 2),
            (f"Bird Final Score: {bird_score:.2f}", (0,0,255), 0.56, 2),
            (f"Drone Final Score: {drone_score:.2f}", (0,255,0), 0.56, 2),
            (f"Score Gap: {score_gap:.2f}", (255,255,255), 0.56, 2),
        ]
        draw_panel(frame, lines)

    else:
        missing_count += 1
        if missing_count <= MISSING_FRAME_HOLD:
            final_label = last_final_label
        else:
            final_label = "No Detection"
            target_track_id = None

        prev_center = None
        prev_speed = None
        prev_angle = None
        track_points.clear()

        lines = [
            ("CROP-RECHECK DRONE/BIRD CLASSIFICATION", (255,255,255), 0.68, 2),
            ("No detection in current frame", (0,0,255), 0.74, 2),
            (f"Holding Previous Label: {final_label}", (0,255,255), 0.60, 2),
            (f"Missing Count: {missing_count}", (220,220,220), 0.56, 1),
        ]
        draw_panel(frame, lines, w=650, h=170)

    out.write(frame)
    prev_gray = curr_gray.copy()

# =========================================================
# 7. FINAL RESULT
# =========================================================
cap.release()
out.release()

filtered = [x for x in video_labels if x in ["Bird", "Drone", "Uncertain"]]
if filtered:
    counts = dict(Counter(filtered))
    final_video_label = Counter(filtered).most_common(1)[0][0]
else:
    counts = {}
    final_video_label = "Unknown"

print("\nDone ✅")
print("Saved at:", output_path)
print("Total frames processed:", frame_count)
print("Final video classification:", final_video_label)
print("Video label distribution:", counts)
print("YOLO Majority:", majority_vote(yolo_vote_hist))
print("Crop Majority:", majority_vote(recheck_vote_hist))

FPS: 29.97002997002997
Frame Size: 1920 x 1080
🚨 DRONE DETECTED

Done ✅
Saved at: /kaggle/working/final_crop_recheck_output.mp4
Total frames processed: 428
Final video classification: Drone
Video label distribution: {'Drone': 428}
YOLO Majority: Drone
Crop Majority: Drone


In [ ]:
import zipfile

zip_path = "/kaggle/working/output_video.zip"
video_path = "/kaggle/working/final_crop_recheck_output.mp4"

with zipfile.ZipFile(zip_path, 'w') as z:
    z.write(video_path, arcname="output.mp4")

print("ZIP ready:", zip_path)

In [ ]:
# import cv2

# input_path = "/kaggle/working/final_crop_recheck_output.mp4"
# output_path = "/kaggle/working/final_small.mp4"

# cap = cv2.VideoCapture(input_path)

# # aggressive compression
# fps = 10
# width = 640
# height = 360

# fourcc = cv2.VideoWriter_fourcc(*"mp4v")
# out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame = cv2.resize(frame, (width, height))
#     out.write(frame)

# cap.release()
# out.release()

# print("Compressed:", output_path)